# Create co-occurence graph

Steps applied in this notebook

1. Software mentions curated as "not a software" were removed.
2. Mentions without a "mapped to software" entry (i.e., marked as "not\_disambiguated") were normalized to their own software mention.
3. Software mappings with NaN and None values were removed.
4. Only papers that were classified using the open alex dataset were used (see notebook `01_publication_classification` on how these results were obtained). 
5. Papers in which software did not cooccur were removed
6. A cooccurence graph was generated from the remaining software mentions. 

Described in the paper as follows:

For preprocessing, three steps were applied: (1) software mentions curated as "not a software" were removed; (2) NaN and None values were removed; and (3) rows that were unable to disambiguated (i.e., marked as "not\_disambiguated" for "mapped\_to\_software") were normalized to their own software mention. This is the reason why the 'Number of unique disambiguated mentions' row of the 'valid\_doi' column has greatly increased from 97,662 to 891,113.

The co-occurrence graph was constructed by grouping the CZI data by DOI and aggregating on the "mapped to software" field, producing a per-DOI list of software. These lists were then sorted, and a hashmap was used to track all observed software pairs and their frequencies. Iterating over all papers, every pairwise tuple and its frequency was recorded, and this hashmap was subsequently used to construct the co-occurrence graph. Each software vertex was then annotated with the DOIs in which that software was mentioned, along with the total number of papers. Software vertices that did not connect to any other software were removed.

1. [Notebook Setup](#setup)
2. [Cleaning](#cleaning)
    1. [Software Mapping](#normalize)
    2. [Metrics of the cleaning process](#metrics)
    3. [Apply cleaning](#apply_clean)
    4. [Save doi set](#doi_set)
3. [Create cooccurence graph](#graph)
4. [Save cooccurence graph](#save)
5. [How to Load cooccurence graph](#load)

<a id='setup'></a>

## Notebook setup
required packages and directory specifications
The notebook assumes that the dataset files are stored in a folder `data` that sits as the same level as the `notebooks` directory. The assumed directory structure is the following:

- `notebooks`
-  `data` 
    - `occurence_graph`
        - metrics.txt
        - agg_df.csv.gz
        - graph_joblib.pkl.gz

In [2]:
import pandas as pd
import numpy as np
import json
from tqdm import tqdm
from pathlib import Path
import plotly.express as px
from itertools import combinations
from collections import defaultdict
import igraph as ig
import gzip
from pathlib import Path
import joblib
import ast

pd.set_option('max_colwidth', 1000)

In [ ]:
ROOT_OCCURRENCE_GRAPH = Path("../data/occurence_graph/")

agg_df = pd.read_csv(ROOT_OCCURRENCE_GRAPH / 'agg_df.csv.gz', compression="gzip")

# convert software_list to a list
agg_df['software_list'] = agg_df['software_list'].apply(
    lambda x: ast.literal_eval(x) if pd.notna(x) else []
)

<a id='graph'></a>
# Create cooccurence graph

A network is created using the unique mentions (i.e., the software) as vertices and the edge between two papers indicates they have appeared together in the same paper. The weight indicates how often these two software cooccur across all papers.
Every software also has every paper it occurs in attached in with the `dois` label. Thus calling the `dois` attribute of a software node returns all the papers that software was found in.  

Every node has the `doi_count` attribute which is simply the number of papers it occured in. 

In [4]:
for software_list in agg_df['software_list']:
    print (software_list)
    print (type(software_list))
    break

['Stata', 'R']
<class 'list'>


In [ ]:
from itertools import combinations
from collections import defaultdict

# map keeping track of co-occurences
cooccurrence = defaultdict(int)

for software_list in agg_df['software_list']:
    # get all the software pairs from the paper
    for pair in combinations(sorted(software_list), 2):
        cooccurrence[pair] += 1

# convert to list of tuples for inserting into igrahp [(software_1: string, software_2: string, frequency: int)] 
edges = [(s1, s2, freq) for (s1, s2), freq in cooccurrence.items()]

In [6]:
import igraph as ig

g = ig.Graph()

# add all software as vertices (nodes)
all_software = list({s for software_list in agg_df['software_list'] for s in software_list})
g.add_vertices(all_software)

# add weighted edges
g.add_edges([(s1, s2) for s1, s2, _ in edges])
g.es['weight'] = [freq for _, _, freq in edges]

In [ ]:
from collections import defaultdict

# software -> [doi, doi, ...] mapping 
software_dois = defaultdict(list)

for _, row in agg_df.iterrows():
    for software in row['software_list']:
        software_dois[software].append(row['doi'])

# attach the papers to the to vertices as metadata
g.vs['dois'] = [software_dois[v['name']] for v in g.vs]

In [8]:
# count the number of doi the node has
g.vs['doi_count'] = [len(set(software_dois[v['name']])) for v in g.vs]

<a id='save'></a>
# Save cooccurence graph

In [9]:
import gzip
from pathlib import Path
out_dir = Path("../data/occurence_graph/")

In [10]:
joblib.dump(g, out_dir / 'graph_joblib.pkl.gz')

['..\\data\\occurence_graph\\graph_joblib.pkl.gz']